# Requirements

To run this notebook, you will need:
- Python 3.12 and the following dependencies:
    - amplpy
    - cplex
    - notebook/ipykernel
- AMPL + CPLEX

In [22]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [1]:
import pandas as pd

from lower_bounds import (
    arcs_as_dataframe,
    export_arcs_to_csv,
    find_gomory_cut,
    sec_identification,
    find_2match,
    solve_relaxation,
    solve_ilp,
    clear_cuts,
    add_cut,
    is_circuit,
    connected_components,
)

In [2]:
clear_cuts()
summary_data = []

In [3]:
def summary_entry(name, arcs, cost) -> list:
    return [
        name,
        cost,
        sum(not v.is_integer() for v in arcs.values()),
        connected_components(arcs),
        is_circuit(arcs),
    ]

## a) Linear relaxation

In [4]:
lr_arcs, lr_cost = solve_relaxation()
lr_df = arcs_as_dataframe(lr_arcs)
export_arcs_to_csv(lr_arcs, "./data/relaxation.csv")
summary_data.append(summary_entry("Base Relaxation", lr_arcs, lr_cost))
lr_df[lr_df["value"] > 0]

[ERROR] 
	Error executing "solve" command:
	error processing param c:
		276 invalid subscripts discarded:
		c[1,1]
		c[2,1]
		c[2,2]
		and 273 more.


CPLEX 22.1.2: CPLEX 22.1.2: optimal solution; objective 6043
33 simplex iterations


,i,j,value
1,1,3,0.5
5,1,7,0.5
16,1,18,1.0
29,2,10,0.5
30,2,11,0.5
38,2,19,1.0
46,3,7,0.5
59,3,20,1.0
64,4,6,0.5
68,4,10,1.0


## b) SEC

In [5]:
sec_cut = sec_identification(lr_arcs)
add_cut(sec_cut, "SEC")

Merging node 1 ({1}) and node 18 ({18}) into node A
Subtour found between nodes {1, 18, 7}
x[1, 7] + x[1, 18] + x[7, 18] = 2.5 !<= 2


In [6]:
sec_arcs, sec_cost = solve_relaxation()
sec_df = arcs_as_dataframe(sec_arcs)
export_arcs_to_csv(sec_arcs, "./data/sec_1.csv")
summary_data.append(summary_entry("SEC 1", sec_arcs, sec_cost))
sec_df[sec_df["value"] > 0]

[ERROR] 
	Error executing "solve" command:
	error processing param c:
		276 invalid subscripts discarded:
		c[1,1]
		c[2,1]
		c[2,2]
		and 273 more.


CPLEX 22.1.2: CPLEX 22.1.2: optimal solution; objective 6146.5
38 simplex iterations


,i,j,value
15,1,17,1.0
16,1,18,1.0
29,2,10,0.5
30,2,11,0.5
38,2,19,1.0
46,3,7,1.0
59,3,20,1.0
64,4,6,0.5
68,4,10,1.0
69,4,11,0.5


In [7]:
sec_cut_2 = sec_identification(sec_arcs)
add_cut(sec_cut_2, "SEC")

Merging node 1 ({1}) and node 17 ({17}) into node A
Merging node 18 ({18}) and node A ({1, 17}) into node B
Merging node 7 ({7}) and node B ({1, 18, 17}) into node C
Merging node 3 ({3}) and node C ({1, 18, 17, 7}) into node D
Merging node 20 ({20}) and node D ({1, 18, 3, 17, 7}) into node E
Merging node 14 ({14}) and node E ({1, 18, 3, 20, 17, 7}) into node F
Merging node 8 ({8}) and node F ({1, 18, 3, 20, 17, 7, 14}) into node G
Merging node 5 ({5}) and node G ({1, 3, 7, 8, 14, 17, 18, 20}) into node H
Merging node 21 ({21}) and node H ({1, 3, 5, 7, 8, 14, 17, 18, 20}) into node I
Merging node 22 ({22}) and node I ({1, 3, 5, 7, 8, 14, 17, 18, 20, 21}) into node J
Merging node 13 ({13}) and node J ({1, 3, 5, 7, 8, 14, 17, 18, 20, 21, 22}) into node K
Merging node 9 ({9}) and node K ({1, 3, 5, 7, 8, 13, 14, 17, 18, 20, 21, 22}) into node L
Merging node 19 ({19}) and node L ({1, 3, 5, 7, 8, 9, 13, 14, 17, 18, 20, 21, 22}) into node M
Merging node 2 ({2}) and node M ({1, 3, 5, 7, 8, 9, 1

In [8]:
sec_2_arcs, sec_2_cost = solve_relaxation()
sec_2_df = arcs_as_dataframe(sec_2_arcs)
export_arcs_to_csv(sec_2_arcs, "./data/sec_2.csv")
summary_data.append(summary_entry("SEC 2", sec_2_arcs, sec_2_cost))
sec_2_df[sec_2_df["value"] > 0]

[ERROR] 
	Error executing "solve" command:
	error processing param c:
		276 invalid subscripts discarded:
		c[1,1]
		c[2,1]
		c[2,2]
		and 273 more.


CPLEX 22.1.2: CPLEX 22.1.2: optimal solution; objective 6147
37 simplex iterations


,i,j,value
15,1,17,1.0
16,1,18,1.0
29,2,10,0.5
30,2,11,0.5
38,2,19,1.0
46,3,7,1.0
59,3,20,1.0
64,4,6,1.0
68,4,10,1.0
84,5,8,1.0


## c) 2-matching Cut

In [9]:
match_cut = find_2match(sec_2_arcs)
add_cut(match_cut, "2Matching")

(x[2, 10] + x[2, 11] + x[10, 11]) + x[2, 19] + x[4, 10] + x[6, 11] = 4.5 !<= 4


In [10]:
match_arcs, match_cost = solve_relaxation()
export_arcs_to_csv(match_arcs, "./data/match_1.csv")
summary_data.append(summary_entry("2-Matching Cut 1", match_arcs, match_cost))

[ERROR] 
	Error executing "solve" command:
	error processing param c:
		276 invalid subscripts discarded:
		c[1,1]
		c[2,1]
		c[2,2]
		and 273 more.


CPLEX 22.1.2: CPLEX 22.1.2: optimal solution; objective 6148.5
37 simplex iterations


In [11]:
match_cut_2 = find_2match(match_arcs)
add_cut(match_cut_2, "2Matching")

(x[12, 16] + x[12, 21] + x[16, 21]) + x[12, 15] + x[16, 23] + x[17, 21] = 4.5 !<= 4


In [12]:
match_2_arcs, match_2_cost = solve_relaxation()
export_arcs_to_csv(match_2_arcs, "./data/match_2.csv")
summary_data.append(summary_entry("2-Matching Cut 2", match_2_arcs, match_2_cost))

[ERROR] 
	Error executing "solve" command:
	error processing param c:
		276 invalid subscripts discarded:
		c[1,1]
		c[2,1]
		c[2,2]
		and 273 more.


CPLEX 22.1.2: CPLEX 22.1.2: optimal solution; objective 6149.5
37 simplex iterations


In [13]:
match_2_df = arcs_as_dataframe(match_2_arcs)
match_2_df[match_2_df["value"] > 0]

,i,j,value
15,1,17,1.0
16,1,18,1.0
30,2,11,1.0
38,2,19,1.0
46,3,7,1.0
59,3,20,1.0
64,4,6,1.0
68,4,10,1.0
84,5,8,1.0
98,5,22,1.0


## d) Gomory Cut

In [14]:
gomory_cuts = find_gomory_cut()
for cut in gomory_cuts:
    add_cut(cut, "Gomory")

[ERROR] 
	Error executing "write" command:
	error processing param c:
		276 invalid subscripts discarded:
		c[1,1]
		c[2,1]
		c[2,2]
		and 273 more.


Reading AMPL model
Exporting AMPL model
Loading model in CPLEX

Selected objective sense:  MINIMIZE
Selected objective  name:  R0028
Selected RHS        name:  B
Selected bound      name:  BOUND
Version identifier: 22.1.2.0 | 2024-12-09 | 8bd2200c8
CPXPARAM_Read_DataCheck                          1
Tried aggregator 1 time.
No LP presolve or aggregator reductions.
Presolve time = 0.00 sec. (0.15 ticks)
Initializing dual steep norms . . .

Iteration log . . .
Iteration:     1   Dual objective     =           507.000000
x[1, 17],1.0
x[4, 11],0.0
x[6, 11],0.5
x[12, 15],0.5
x[1, 7],0.0
x[2, 19],1.0
x[3, 7],1.0
x[10, 11],0.5
x[5, 8],1.0
x[4, 6],1.0
x[7, 20],0.0
x[2, 9],0.0
x[9, 19],1.0
x[6, 10],0.5
x[2, 11],1.0
x[12, 16],0.5
x[7, 14],0.0
x[5, 14],0.0
x[15, 23],1.0
x[15, 16],0.5
x[12, 17],0.0
x[7, 18],1.0
x[13, 19],0.0
x[1, 3],0.0
x[12, 21],1.0
x[13, 22],1.0
x[16, 21],0.0
Ignoring integer basic variable
Ignoring integer basic variable
Selected variable x[6, 11] C0105 (tableau index 2) with va

In [15]:
gomory_arcs, gomory_cost = solve_relaxation()
export_arcs_to_csv(gomory_arcs, "./data/gomory.csv")
summary_data.append(summary_entry("Gomory Cuts", gomory_arcs, gomory_cost))

[ERROR] 
	Error executing "solve" command:
	error processing param c:
		276 invalid subscripts discarded:
		c[1,1]
		c[2,1]
		c[2,2]
		and 273 more.


CPLEX 22.1.2: CPLEX 22.1.2: optimal solution; objective 6363
39 simplex iterations


In [16]:
gomory_df = arcs_as_dataframe(gomory_arcs)
gomory_df[gomory_df["value"] > 0]

,i,j,value
5,1,7,1.0
16,1,18,1.0
30,2,11,1.0
38,2,19,1.0
46,3,7,1.0
59,3,20,1.0
64,4,6,1.0
68,4,10,0.5
69,4,11,0.5
89,5,13,1.0


## e) Branch-and-Bound

In [17]:
integer_arcs, integer_cost = solve_ilp()
export_arcs_to_csv(integer_arcs, "./data/ilp.csv")
summary_data.append(summary_entry("Branch & Bound", integer_arcs, integer_cost))

[ERROR] 
	Error executing "solve" command:
	error processing param c:
		276 invalid subscripts discarded:
		c[1,1]
		c[2,1]
		c[2,2]
		and 273 more.


CPLEX 22.1.2: CPLEX 22.1.2: optimal solution; objective 6387
49 simplex iterations


In [18]:
integer_df = arcs_as_dataframe(integer_arcs)
integer_df[integer_df["value"] > 0]

,i,j,value
5,1,7,1.0
16,1,18,1.0
23,2,4,1.0
30,2,11,1.0
46,3,7,1.0
59,3,20,1.0
68,4,10,1.0
89,5,13,1.0
98,5,22,1.0
103,6,10,1.0


## Summary

In [19]:
df = pd.DataFrame(
    data=summary_data,
    columns=["Name", "Cost", "# Fractional arcs", "# Connected Components", "Circuit"],
)
df.to_csv("./data/summary.csv", index=False)
df

,Name,Cost,# Fractional arcs,# Connected Components,Circuit
0,Base Relaxation,6043.0,8,2,False
1,SEC 1,6146.5,8,1,False
2,SEC 2,6147.0,6,1,False
3,2-Matching Cut 1,6148.5,6,1,False
4,2-Matching Cut 2,6149.5,6,1,False
5,Gomory Cuts,6363.0,6,2,False
6,Branch & Bound,6387.0,0,3,False


# Tests

In [ ]:
arcs = {
    (1, 2): 1,
    (1, 3): 5 / 12,
    (1, 4): 5 / 12,
    (1, 5): 1 / 6,
    (2, 3): 5 / 12,
    (2, 4): 5 / 12,
    (2, 6): 1 / 6,
    (3, 4): 1,
    (3, 7): 1 / 6,
    (4, 8): 1 / 6,
    (5, 6): 1,
    (5, 7): 5 / 12,
    (5, 8): 5 / 12,
    (6, 7): 5 / 12,
    (6, 8): 5 / 12,
    (7, 8): 1,
}

In [ ]:
sec_identification(arcs)

In [ ]:
arcs = {
    (1, 2): 1 / 2,
    (1, 3): 1 / 2,
    (1, 4): 1,
    (2, 3): 1 / 2,
    (2, 5): 1,
    (3, 6): 1,
    (4, 5): 1 / 2,
    (4, 6): 1 / 2,
    (5, 6): 1 / 2,
}

In [ ]:
find_2match(arcs)

In [ ]:
find_gomory_cut(arcs)

In [ ]:
for i in range(1, 24):
    for j in range(i + 1, 24):
        print(f"({i}, {j})", end=" ")
    print("")